# Model Definition: EfficientNet-B0
This notebook defines and compiles the EfficientNet-B0 model for image classification, using the preprocessed dataset.

In [ ]:
# 1. Import Libraries and EfficientNet-B0
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import os

In [ ]:
# 2. Load Preprocessed Dataset
labels = pd.read_csv('labels_preprocessed.csv')
print('Loaded preprocessed labels:', labels.shape)
labels.head()

In [ ]:
# 3. Prepare Data for Training
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Parameters
img_size = 224
batch_size = 32

# Split data
df_train, df_val = train_test_split(labels, test_size=0.2, stratify=labels['label_encoded'], random_state=42)

# Data generators for images
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2,
                             rotation_range=20, width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True)

train_gen = datagen.flow_from_dataframe(
    df_train, directory='train/', x_col='filename', y_col='label_encoded',
    target_size=(img_size, img_size), batch_size=batch_size, class_mode='categorical', subset='training')

val_gen = datagen.flow_from_dataframe(
    df_val, directory='train/', x_col='filename', y_col='label_encoded',
    target_size=(img_size, img_size), batch_size=batch_size, class_mode='categorical', subset='validation')

In [ ]:
# 4. Build EfficientNet-B0 Model
num_classes = labels['label_encoded'].nunique()

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(img_size, img_size, 3))
base_model.trainable = False  # Transfer learning

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.summary()

In [ ]:
# 5. Compile Model
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])
print('Model compiled and ready for training.')